# Role-Conditioned LoRA Benchmark
Upload this notebook to Google Colab and select a **GPU runtime** (`Runtime > Change runtime type > T4 GPU or better`).

This notebook now does both:
- qualitative baseline vs tuned response comparison
- quantitative head-to-head benchmarking against the baseline

For the full benchmark, upload:
- `role_conditioned_dpo_lora.zip`
- one scenario file such as `evaluated.jsonl`, `test_scenarios_holdout.jsonl`, or `test_scenarios.jsonl`

The automated scoring step uses an OpenRouter judge model, so you will also need an `OPENROUTER_API_KEY`.


In [ ]:
# Step 1: Install dependencies
!pip install -q "transformers>=4.40" "peft>=0.10" "accelerate>=0.28" "bitsandbytes>=0.43" "openai>=1.0" pandas sentencepiece

import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Switch Colab to a GPU runtime before continuing.')


In [ ]:
# Step 2: Upload files
# Required: role_conditioned_dpo_lora.zip
# Recommended for benchmarking: evaluated.jsonl or test_scenarios_holdout.jsonl
from google.colab import files

print('Upload role_conditioned_dpo_lora.zip and optionally one scenario JSONL file')
uploaded = files.upload()

for name, content in uploaded.items():
    with open(name, 'wb') as f:
        f.write(content)

print('Saved files:', list(uploaded.keys()))


In [ ]:
# Step 2 (Alternative): Mount Google Drive instead of uploading
# Uncomment if your files are stored in Drive.

# from google.colab import drive
# drive.mount('/content/drive')
# ADAPTER_ZIP = '/content/drive/MyDrive/cs288/role_conditioned_dpo_lora.zip'
# SCENARIO_PATH = '/content/drive/MyDrive/cs288/evaluated.jsonl' 


In [ ]:
# Step 3: Helpers
import gc
import json
import os
import random
import tempfile
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from openai import OpenAI
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

PROMPT_TEMPLATE = """You are in the following organizational role:
{role_description}

Interaction context:
{context_description}

Situation: {situation}
Background: {background}

Respond appropriately given your role and the situation."""

JUDGE_COMPARISON_PROMPT = """You are an expert organizational behavior evaluator. You will evaluate two responses to the same workplace scenario. You do NOT know which response comes from which model.

## Scenario
**Speaker's role:** {role_description}
**Interaction context:** {context_description}
**Situation:** {situation}
**Background:** {background}

## Response A:
{response_a}

## Response B:
{response_b}

## Evaluation
Rate each response on these dimensions (1-5 scale):

1. **Appropriateness**: Does the tone, formality, and content match the speaker's hierarchical position and power relationship?
2. **Role Fidelity**: Does the response reflect the speaker's hierarchy level, functional domain, and seniority accurately?
3. **Collaborative Realism**: Does the response exhibit realistic collaborative characteristics (power-sensitive communication, decision authority awareness, accountability language)?

Then determine which response is better overall, or if they are tied.

Output as JSON:
{
    "response_a": {
        "appropriateness": <1-5>,
        "role_fidelity": <1-5>,
        "collaborative_realism": <1-5>,
        "rationale": "<brief explanation>"
    },
    "response_b": {
        "appropriateness": <1-5>,
        "role_fidelity": <1-5>,
        "collaborative_realism": <1-5>,
        "rationale": "<brief explanation>"
    },
    "winner": "<A or B or tie>",
    "winner_rationale": "<why this response is better>"
}"""

DEFAULT_SCENARIO = {
    'role_description': (
        'You are a C-level executive in operations, who has no direct authority '
        'over the person they are speaking with (cross-functional). You are a '
        'long-tenured employee with deep institutional knowledge.'
    ),
    'context_description': (
        'You are mentoring a more junior colleague. Provide guidance, share '
        'experience, and support their development.'
    ),
    'situation': (
        "Sarah Chen, a Senior Data Analyst in the Marketing department, presented "
        "a forecast to the cross-functional project team for the 'Evergreen "
        "Initiative' that was significantly more pessimistic than previous "
        "models. You offered to meet with her to understand her methodology and "
        "share insights from your years at the company."
    ),
    'background': (
        "The 'Evergreen Initiative' is a company-wide sustainability effort with "
        "a major Q4 launch. Sarah's forecast could change marketing budget "
        "allocation and project strategy. Previous forecasts were more "
        "optimistic, but Sarah incorporated external consumer-sentiment signals. "
        "You have been at the company for 18 years and have seen similar "
        "initiatives succeed or stall depending on how the data is interpreted."
    ),
}

SCORE_DIMS = ['appropriateness', 'role_fidelity', 'collaborative_realism']


def pick_existing_path(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    return None


def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_prompt(scenario):
    return PROMPT_TEMPLATE.format(
        role_description=scenario['role_description'],
        context_description=scenario['context_description'],
        situation=scenario['situation'],
        background=scenario['background'],
    )


def load_scenarios(path=None, limit=None, holdout_only=True, seed=42):
    if path is None:
        return [DEFAULT_SCENARIO]

    rows = []
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            required = all(
                key in row for key in ['role_description', 'context_description', 'situation', 'background']
            )
            if required:
                rows.append(row)

    if not rows:
        raise ValueError(f'No usable scenarios found in {path}')

    if holdout_only and any('split' in row for row in rows):
        holdout_rows = [row for row in rows if row.get('split') == 'holdout']
        if holdout_rows:
            rows = holdout_rows

    if limit is not None and len(rows) > limit:
        sampler = random.Random(seed)
        rows = sampler.sample(rows, limit)

    return rows


def find_adapter_dir(root):
    root = Path(root)
    if (root / 'adapter_config.json').exists():
        return root
    for candidate in root.rglob('adapter_config.json'):
        return candidate.parent
    raise FileNotFoundError(
        f'Could not find adapter_config.json under {root}. '
        'Make sure you uploaded the correct LoRA adapter zip.'
    )


def extract_adapter(adapter_zip):
    temp_dir = Path(tempfile.mkdtemp(prefix='colab_lora_'))
    with zipfile.ZipFile(adapter_zip) as zf:
        zf.extractall(temp_dir)
    return find_adapter_dir(temp_dir), temp_dir


def load_tokenizer(base_model_name, adapter_dir):
    tokenizer_source = adapter_dir if (adapter_dir / 'tokenizer_config.json').exists() else base_model_name
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_base_model(base_model_name):
    quant_config = None
    torch_dtype = torch.float32
    if torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        torch_dtype = torch.bfloat16

    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch_dtype,
        device_map='auto',
        quantization_config=quant_config,
    )
    model.eval()
    return model


def generate_response(model, tokenizer, prompt, max_new_tokens=220, temperature=0.0, seed=42):
    if temperature > 0:
        set_seed(seed)

    inputs = tokenizer(prompt, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    generation_kwargs = {
        'max_new_tokens': max_new_tokens,
        'pad_token_id': tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs.update({
            'temperature': temperature,
            'do_sample': True,
            'top_p': 0.95,
        })
    else:
        generation_kwargs.update({'do_sample': False})

    with torch.inference_mode():
        outputs = model.generate(**inputs, **generation_kwargs)

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def run_local_comparison(scenarios, base_model_name, adapter_dir, max_new_tokens=220, temperature=0.0, seed=42):
    tokenizer = load_tokenizer(base_model_name, adapter_dir)
    base_model = load_base_model(base_model_name)

    rows = []
    print(f'Generating baseline responses for {len(scenarios)} scenarios...')
    for idx, scenario in enumerate(scenarios):
        prompt = build_prompt(scenario)
        baseline_response = generate_response(
            base_model,
            tokenizer,
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            seed=seed + idx,
        )
        rows.append({
            'scenario_index': idx,
            'scenario': scenario,
            'prompt': prompt,
            'baseline_response': baseline_response,
        })
        if (idx + 1) % 5 == 0 or idx + 1 == len(scenarios):
            print(f'  Baseline progress: {idx + 1}/{len(scenarios)}')

    print('Loading adapter and generating tuned responses...')
    tuned_model = PeftModel.from_pretrained(base_model, adapter_dir)
    tuned_model.eval()
    for idx, row in enumerate(rows):
        row['tuned_response'] = generate_response(
            tuned_model,
            tokenizer,
            row['prompt'],
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            seed=seed + idx,
        )
        if (idx + 1) % 5 == 0 or idx + 1 == len(rows):
            print(f'  Tuned progress: {idx + 1}/{len(rows)}')

    del tuned_model
    del base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return rows


def print_comparison_examples(rows, limit=3):
    for row in rows[:limit]:
        scenario = row['scenario']
        print('=' * 100)
        print(f"Example {row['scenario_index'] + 1}")
        print('-' * 100)
        print('Role:', scenario['role_description'])
        print('Context:', scenario['context_description'])
        print('Situation:', scenario['situation'])
        print('Background:', scenario['background'])
        print()
        print('Baseline response:')
        print()
        print(row['baseline_response'])
        print()
        print('Tuned response:')
        print()
        print(row['tuned_response'])
        print()


def strip_json_fences(text):
    text = text.strip()
    if text.startswith('```json'):
        text = text[7:]
    elif text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
    return text.strip()


def ensure_openrouter_key():
    key = os.environ.get('OPENROUTER_API_KEY')
    if key:
        return key
    try:
        from google.colab import userdata
        key = userdata.get('OPENROUTER_API_KEY')
    except Exception:
        key = None
    if key:
        os.environ['OPENROUTER_API_KEY'] = key
    return key


def call_openrouter(prompt, model, system_prompt=None, temperature=0.1):
    api_key = ensure_openrouter_key()
    if not api_key:
        raise ValueError('Set OPENROUTER_API_KEY before running the benchmark judge.')

    client = OpenAI(api_key=api_key, base_url='https://openrouter.ai/api/v1')
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def judge_pairwise(row, judge_model, rng, retries=3, sleep_seconds=0.0):
    scenario = row['scenario']
    if rng.random() < 0.5:
        response_a, response_b = row['baseline_response'], row['tuned_response']
        a_is = 'baseline'
    else:
        response_a, response_b = row['tuned_response'], row['baseline_response']
        a_is = 'tuned'

    prompt = JUDGE_COMPARISON_PROMPT.format(
        role_description=scenario['role_description'],
        context_description=scenario['context_description'],
        situation=scenario['situation'],
        background=scenario['background'],
        response_a=response_a,
        response_b=response_b,
    )

    last_error = None
    for attempt in range(retries):
        try:
            raw = call_openrouter(
                prompt + '\n\nRespond with valid JSON only.',
                model=judge_model,
                temperature=0.1,
            )
            judge_result = json.loads(strip_json_fences(raw))
            break
        except Exception as exc:
            last_error = exc
            if attempt + 1 == retries:
                raise
            time.sleep(max(sleep_seconds, 1.0))
    else:
        raise last_error

    winner_raw = str(judge_result.get('winner', 'tie')).strip().upper()
    if winner_raw == 'A':
        winner = a_is
    elif winner_raw == 'B':
        winner = 'tuned' if a_is == 'baseline' else 'baseline'
    else:
        winner = 'tie'

    response_a_scores = judge_result.get('response_a', {})
    response_b_scores = judge_result.get('response_b', {})
    baseline_scores = response_a_scores if a_is == 'baseline' else response_b_scores
    tuned_scores = response_b_scores if a_is == 'baseline' else response_a_scores

    return {
        **row,
        'judge_result': judge_result,
        'winner': winner,
        'a_is': a_is,
        'baseline_scores': baseline_scores,
        'tuned_scores': tuned_scores,
    }


def run_llm_judge_benchmark(rows, judge_model, seed=42, sleep_seconds=0.0):
    rng = random.Random(seed)
    judged_rows = []
    wins = {'baseline': 0, 'tuned': 0, 'tie': 0}

    print(f'Judging {len(rows)} baseline-vs-tuned pairs with {judge_model}...')
    for idx, row in enumerate(rows):
        judged = judge_pairwise(row, judge_model, rng=rng, sleep_seconds=sleep_seconds)
        judged_rows.append(judged)
        wins[judged['winner']] += 1
        if (idx + 1) % 5 == 0 or idx + 1 == len(rows):
            print(
                f"  Judge progress: {idx + 1}/{len(rows)} | "
                f"baseline={wins['baseline']} tuned={wins['tuned']} tie={wins['tie']}"
            )
    return judged_rows


def summarize_judged_results(rows):
    wins = {'baseline': 0, 'tuned': 0, 'tie': 0}
    score_buckets = {
        'baseline': defaultdict(list),
        'tuned': defaultdict(list),
    }
    flat_rows = []

    for row in rows:
        wins[row['winner']] += 1
        baseline_scores = row.get('baseline_scores', {})
        tuned_scores = row.get('tuned_scores', {})
        for dim in SCORE_DIMS:
            if dim in baseline_scores:
                score_buckets['baseline'][dim].append(baseline_scores[dim])
            if dim in tuned_scores:
                score_buckets['tuned'][dim].append(tuned_scores[dim])

        scenario = row['scenario']
        flat_rows.append({
            'scenario_index': row['scenario_index'],
            'winner': row['winner'],
            'hierarchy': scenario.get('role', {}).get('hierarchy', ''),
            'function': scenario.get('role', {}).get('function', ''),
            'power_relation': scenario.get('role', {}).get('power_relation', ''),
            'appropriateness_baseline': baseline_scores.get('appropriateness'),
            'appropriateness_tuned': tuned_scores.get('appropriateness'),
            'role_fidelity_baseline': baseline_scores.get('role_fidelity'),
            'role_fidelity_tuned': tuned_scores.get('role_fidelity'),
            'collaborative_realism_baseline': baseline_scores.get('collaborative_realism'),
            'collaborative_realism_tuned': tuned_scores.get('collaborative_realism'),
            'winner_rationale': row.get('judge_result', {}).get('winner_rationale', ''),
        })

    total = max(sum(wins.values()), 1)
    mean_scores = {}
    mean_deltas = {}
    for model_name in ['baseline', 'tuned']:
        mean_scores[model_name] = {}
        for dim in SCORE_DIMS:
            values = score_buckets[model_name][dim]
            mean_scores[model_name][dim] = sum(values) / max(len(values), 1)
    for dim in SCORE_DIMS:
        mean_deltas[dim] = mean_scores['tuned'][dim] - mean_scores['baseline'][dim]

    summary = {
        'total_scenarios': sum(wins.values()),
        'wins': wins,
        'win_rates': {key: wins[key] / total for key in wins},
        'win_rate_excluding_ties': wins['tuned'] / max(wins['baseline'] + wins['tuned'], 1),
        'mean_scores': mean_scores,
        'mean_deltas': mean_deltas,
    }
    return summary, pd.DataFrame(flat_rows)


def plot_summary(summary):
    dims = SCORE_DIMS
    baseline_means = [summary['mean_scores']['baseline'][dim] for dim in dims]
    tuned_means = [summary['mean_scores']['tuned'][dim] for dim in dims]
    x = range(len(dims))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    win_order = ['baseline', 'tuned', 'tie']
    win_colors = ['#9aa5b1', '#2b8a3e', '#f08c00']
    axes[0].bar(win_order, [summary['wins'][key] for key in win_order], color=win_colors)
    axes[0].set_title('Head-to-Head Wins')
    axes[0].set_ylabel('Scenarios')

    axes[1].bar([i - width / 2 for i in x], baseline_means, width=width, label='Baseline', color='#9aa5b1')
    axes[1].bar([i + width / 2 for i in x], tuned_means, width=width, label='Tuned', color='#2b8a3e')
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(['Appropriateness', 'Role Fidelity', 'Collaborative Realism'], rotation=15)
    axes[1].set_ylim(0, 5)
    axes[1].set_title('Mean Judge Scores')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def save_jsonl(rows, path):
    with open(path, 'w') as f:
        for row in rows:
            f.write(json.dumps(row) + '\n')


In [ ]:
# Step 4: Configure paths and evaluation settings
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
ADAPTER_ZIP = globals().get('ADAPTER_ZIP', 'role_conditioned_dpo_lora.zip')
SCENARIO_PATH = globals().get(
    'SCENARIO_PATH',
    pick_existing_path([
        'evaluated.jsonl',
        'test_scenarios_holdout.jsonl',
        'test_scenarios.jsonl',
    ])
)

USE_HOLDOUT_ONLY = True
NUM_PREVIEW_EXAMPLES = 3
MAX_EVAL_SCENARIOS = 20

MAX_NEW_TOKENS = 220
TEMPERATURE = 0.0
EVAL_SEED = 42

JUDGE_MODEL = 'google/gemini-2.0-flash-001'
API_SLEEP_SECONDS = 0.0

GENERATED_OUTPUT_PATH = 'generated_model_comparisons.jsonl'
JUDGED_OUTPUT_PATH = 'baseline_vs_tuned_judged.jsonl'
SUMMARY_OUTPUT_PATH = 'baseline_vs_tuned_summary.json'

if not Path(ADAPTER_ZIP).exists():
    raise FileNotFoundError(f'Missing adapter zip: {ADAPTER_ZIP}')

adapter_dir, adapter_temp_dir = extract_adapter(ADAPTER_ZIP)
scenarios = load_scenarios(
    SCENARIO_PATH,
    limit=MAX_EVAL_SCENARIOS,
    holdout_only=USE_HOLDOUT_ONLY,
    seed=EVAL_SEED,
)
preview_scenarios = scenarios[:NUM_PREVIEW_EXAMPLES]

print('Base model:', BASE_MODEL)
print('Adapter dir:', adapter_dir)
print('Scenario file:', SCENARIO_PATH or 'built-in sample only')
print('Scenarios loaded:', len(scenarios))
print('Preview examples:', len(preview_scenarios))
print('Temperature:', TEMPERATURE)
if SCENARIO_PATH is None:
    print('No scenario file found, so preview and benchmark will use the built-in sample scenario.')
elif len(scenarios) < MAX_EVAL_SCENARIOS:
    print('Loaded fewer scenarios than MAX_EVAL_SCENARIOS because the file is smaller or filtered to holdout only.')


In [ ]:
# Step 5: Preview a few baseline vs tuned responses
preview_rows = run_local_comparison(
    preview_scenarios,
    base_model_name=BASE_MODEL,
    adapter_dir=adapter_dir,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    seed=EVAL_SEED,
)
print_comparison_examples(preview_rows, limit=NUM_PREVIEW_EXAMPLES)


In [ ]:
# Step 6: Set your OpenRouter API key for automated evaluation
import getpass

if not ensure_openrouter_key():
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Enter OPENROUTER_API_KEY: ')

print('OPENROUTER_API_KEY loaded:', bool(os.environ.get('OPENROUTER_API_KEY')))


In [ ]:
# Step 7: Run the full baseline vs tuned benchmark
# This cell generates both models' responses, sends them to the judge, saves results, and plots a summary.
comparison_rows = run_local_comparison(
    scenarios,
    base_model_name=BASE_MODEL,
    adapter_dir=adapter_dir,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    seed=EVAL_SEED,
)

save_jsonl(comparison_rows, GENERATED_OUTPUT_PATH)
print(f'Saved generated comparisons to {GENERATED_OUTPUT_PATH}')

judged_rows = run_llm_judge_benchmark(
    comparison_rows,
    judge_model=JUDGE_MODEL,
    seed=EVAL_SEED,
    sleep_seconds=API_SLEEP_SECONDS,
)
save_jsonl(judged_rows, JUDGED_OUTPUT_PATH)
print(f'Saved judged results to {JUDGED_OUTPUT_PATH}')

summary, results_df = summarize_judged_results(judged_rows)
with open(SUMMARY_OUTPUT_PATH, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved summary to {SUMMARY_OUTPUT_PATH}')

print(json.dumps(summary, indent=2))
display(results_df.head(10))
plot_summary(summary)


In [ ]:
# Optional: Download the benchmark outputs
from google.colab import files

for path in [GENERATED_OUTPUT_PATH, JUDGED_OUTPUT_PATH, SUMMARY_OUTPUT_PATH]:
    if Path(path).exists():
        print('Downloading', path)
        files.download(path)


In [ ]:
# Optional: Edit this custom scenario and generate one fresh tuned response
custom_rows = run_local_comparison(
    [
        {
            'role_description': 'You are a middle manager in engineering, who outranks the person they are speaking with. You have 1-5 years at the company.',
            'context_description': 'You are assigning a task to someone. Describe the task clearly, set expectations, and provide any necessary context.',
            'situation': 'The team needs to migrate the authentication service from the legacy monolith to a microservice architecture before the Q3 deadline.',
            'background': 'The migration has been planned for months but keeps getting deprioritized. The engineer you are assigning this to is capable but already has a full plate.',
        }
    ],
    base_model_name=BASE_MODEL,
    adapter_dir=adapter_dir,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    seed=EVAL_SEED,
)
print_comparison_examples(custom_rows, limit=1)
